In [1]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import lightkurve as lk
from lightkurve import search_lightcurve


/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [2]:
def fetchStitchedLC(kic):
    """
    Given a KIC, fetch its light curve data using Lightkurve.
    Returns a stitched LC object.
    """
    kic = int(kic)
    search = lk.search_lightcurve(
        f"KIC {kic}",
        mission="Kepler",
        author="Kepler",
        cadence="long"
    )
    if len(search) == 0:
        raise ValueError(f"No lightcurve found for KIC {kic}")
    return search.download_all().stitch().remove_nans()

def getLCArrays(kic):
    lc = fetchStitchedLC(kic)

    return pd.Series({
        "time": lc.time.value,
        "flux": lc.flux.value,
        "flux_err": lc.flux_err.value if lc.flux_err is not None else None,
        "n_points": len(lc),
    })

def _safe_get_lc_arrays(kic):
    try:
        s = getLCArrays(kic)
        s["KIC"] = int(kic)
        s["error"] = None
        return s
    except Exception as e:
        return pd.Series({
            "KIC": int(kic),
            "time": None,
            "flux": None,
            "flux_err": None,
            "n_points": 0,
            "error": str(e),
        })

def parallel_fetch_lightcurves(df, kic_col=None, max_workers=8, chunksize=100):
    """
    Download light curves in parallel and join them back to df.

    Parameters
    ----------
    df : pd.DataFrame
    kic_col : str or None
        If provided, KICs are read from this column.
        If None, df.index is used.
    """
    if kic_col is not None:
        kics = df[kic_col].to_list()
    else:
        # Only use the index if it is truly the KIC index.
        if df.index.name is None:
            raise ValueError(
                "df.index has no name, so it may not contain KICs. "
                "Pass kic_col='KIC' explicitly."
            )
        kics = df.index.to_list()

    results = []

    for start in range(0, len(kics), chunksize):
        batch = kics[start:start + chunksize]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(_safe_get_lc_arrays, k): k for k in batch}

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"KICs {start + 1}-{start + len(batch)}",
                leave=True
            ):
                results.append(future.result())

    lc_df = pd.DataFrame(results).set_index("KIC")

    if kic_col is not None:
        out = df.copy()
        out["KIC"] = out[kic_col].astype(int)
        out = out.join(lc_df, on="KIC", how="left")
        return out.drop(columns=["KIC"])
    else:
        return df.join(lc_df, how="left")


In [3]:
df = pd.read_csv(r"../assets/data/kepler-eclipsing-binary-catalog.csv")
df = df.drop(columns=["Unnamed: 11"])
df = df.head(290)
df = df.set_index("KIC")


In [4]:
df


,period,period_err,bjd0,bjd0_err,morph,GLon,GLat,kmag,Teff,SC
KIC,,,,,,,,,,
3863594,0.053268,0.000000e+00,55000.000000,0.004327,0.79,-1.0000,-1.0000,-1.000,-1.0,False
10417986,0.073731,0.000000e+00,55000.027476,0.004231,0.99,81.0390,11.0820,9.128,-1.0,True
8912468,0.094838,0.000000e+00,54953.576945,0.005326,0.98,80.1095,7.8882,11.751,6194.0,False
8758716,0.107205,0.000000e+00,54953.672989,0.006197,1.00,77.7478,11.6565,13.531,-1.0,False
10855535,0.112782,0.000000e+00,54964.629315,0.006374,0.99,79.3949,15.9212,13.870,7555.0,False
...,...,...,...,...,...,...,...,...,...,...
5620981,0.333516,2.000000e-07,54964.759352,0.020202,0.79,73.4649,10.9580,14.873,5406.0,False
11502581,0.333636,2.000000e-07,54964.875051,0.017034,0.74,80.6265,16.3835,14.661,5696.0,False
9097798,0.334065,2.000000e-07,54964.721156,0.019209,0.99,78.5825,11.2266,14.581,5592.0,False


In [ ]:
# If KIC is the index:
df_with_lc = parallel_fetch_lightcurves(df, max_workers=8)

# If KIC is in a column:
# df_with_lc = parallel_fetch_lightcurves(df, max_workers=8, kic_col="KIC")


KICs 1-100:  73%|███████▎  | 73/100 [00:33<00:13,  2.07it/s]

In [5]:
df_with_lc


,KIC,period,period_err,bjd0,bjd0_err,morph,GLon,GLat,kmag,Teff,SC,time,flux,flux_err,n_points,error
0,3863594,0.053268,0.000000e+00,55000.000000,0.004327,0.79,-1.0000,-1.0000,-1.000,-1.0,False,NaN,NaN,NaN,NaN,NaN
1,10417986,0.073731,0.000000e+00,55000.027476,0.004231,0.99,81.0390,11.0820,9.128,-1.0,True,NaN,NaN,NaN,NaN,NaN
2,8912468,0.094838,0.000000e+00,54953.576945,0.005326,0.98,80.1095,7.8882,11.751,6194.0,False,NaN,NaN,NaN,NaN,NaN
3,8758716,0.107205,0.000000e+00,54953.672989,0.006197,1.00,77.7478,11.6565,13.531,-1.0,False,NaN,NaN,NaN,NaN,NaN
4,10855535,0.112782,0.000000e+00,54964.629315,0.006374,0.99,79.3949,15.9212,13.870,7555.0,False,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,5620981,0.333516,2.000000e-07,54964.759352,0.020202,0.79,73.4649,10.9580,14.873,5406.0,False,NaN,NaN,NaN,NaN,NaN
286,11502581,0.333636,2.000000e-07,54964.875051,0.017034,0.74,80.6265,16.3835,14.661,5696.0,False,NaN,NaN,NaN,NaN,NaN
287,9097798,0.334065,2.000000e-07,54964.721156,0.019209,0.99,78.5825,11.2266,14.581,5592.0,False,NaN,NaN,NaN,NaN,NaN
288,4945857,0.335415,2.000000e-07,54964.832845,0.018084,0.75,74.5546,7.1707,13.964,5232.0,False,NaN,NaN,NaN,NaN,NaN
